In [ ]:
# ============================================================================
#  SAR RDA (Range Doppler Algorithm) 点目标仿真代码
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt

# 设置 Matplotlib 支持中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ====================================================================
# 1. 定义参数
# ====================================================================
R_nc = 20e3                 # 景中心斜距
Vr = 150                    # 雷达有效速度
Tr = 2.5e-6                 # 发射脉冲时宽
Kr = 20e12                  # 距离调频率
f0 = 5.3e9                  # 雷达工作频率
BW_dop = 80                 # 多普勒带宽
Fr = 60e6                   # 距离采样率
Fa = 200                    # 方位采样率
Naz = 1024                  # 距离线数（即数据矩阵，行数）
Nrg = 320                   # 距离线采样点数（即数据矩阵，列数）
sita_r_c = (3.5 * np.pi) / 180  # 波束斜视角，3.5 度转换为弧度
c = 3e8                     # 光速

In [ ]:
R0 = R_nc * np.cos(sita_r_c)    # 与R_nc相对应的最近斜距，记为R0
Nr = int(Tr * Fr)               # 线性调频信号采样点数
BW_range = Kr * Tr              # 距离向带宽
lamda = c / f0                  # 雷达波长 λ = c / f0                  
fnc = 2 * Vr * np.sin(sita_r_c) / lamda  # 多普勒中心频率 f_dc = 2Vsinθ / λ  
La_real = 0.886 * 2 * Vr * np.cos(sita_r_c) / BW_dop  # 方位向天线长度
beta_bw = 0.886 * lamda / La_real        # 雷达3dB波束宽度      
La = 0.886 * R_nc * lamda / La_real      # 合成孔径长度

NFFT_r = Nrg                # 距离向FFT长度
NFFT_a = Naz                # 方位向FFT长度

### 派生参数计算公式说明
根据基础雷达参数，可以推导出后续处理所需的关键物理量：
* **最近斜距 ($R_0$)**: 景中心斜距在零多普勒方向上的投影，公式为 $R_0 = R_{nc} \cos(\theta_{r,c})$。
* **脉冲采样点数 ($N_r$)**: 发射脉冲在离散时间上的点数，$N_r = T_r \cdot F_r$。
* **距离向带宽 ($BW_{range}$)**: 线性调频信号的总带宽，$BW_{range} = K_r \cdot T_r$。
* **雷达波长 ($\lambda$)**: $\lambda = c / f_0$。
* **多普勒中心频率 ($f_{nc}$)**: 由于斜视角 $\theta_{r,c}$ 的存在，多普勒历程的中心发生偏移，公式为 $f_{nc} = \frac{2 V_r \sin(\theta_{r,c})}{\lambda}$。
* **方位向天线长度 ($L_{a\_real}$)**: 由多普勒带宽反推的物理天线长度，$L_{a\_real} = \frac{0.886 \cdot 2 V_r \cos(\theta_{r,c})}{BW_{dop}}$。
* **雷达波束宽度 ($\beta_{bw}$)**: $\beta_{bw} = \frac{0.886 \cdot \lambda}{L_{a\_real}}$。
* **合成孔径长度 ($L_a$)**: 雷达波束扫过目标的总距离，$L_a = \frac{0.886 \cdot R_{nc} \cdot \lambda}{L_{a\_real}}$。

In [ ]:
# ====================================================================
# 2. 设定仿真点目标的位置
# ====================================================================
delta_R0 = 0       # 目标1方位向时间零点
delta_R1 = 120      # 目标1和目标2的方位向距离差，120m
delta_R2 = 50       # 目标2和目标3的距离向距离差，50m

# 目标1
x1 = R0 
y1 = delta_R0 + x1 * np.tan(sita_r_c)
# 目标2
x2 = x1 
y2 = y1 + delta_R1 
# 目标3
x3 = x2 + delta_R2 
y3 = y2 + delta_R2 * np.tan(sita_r_c) 

x_range = np.array([x1, x2, x3])
y_azimuth = np.array([y1, y2, y3])
nc_target = (y_azimuth - x_range * np.tan(sita_r_c)) / Vr

# ====================================================================
# 3. 距离/方位向时间与频率相关定义
# ====================================================================
tr = 2 * x1 / c + (np.arange(-Nrg/2, Nrg/2)) / Fr     # 距离时间轴
ta = np.arange(-Naz/2, Naz/2) / Fa                    # 方位时间轴

tr_mtx, ta_mtx = np.meshgrid(tr, ta)                  # 时间网格

### 波束中心穿越时刻 (Beam Center Crossing Time)
代码 `nc_target = (y_azimuth - x_range * np.tan(sita_r_c)) / Vr` 计算的是**波束中心穿越时刻 $\eta_c$**。

在斜视模式下，雷达波束是向前（或向后）倾斜 $\theta_{r,c}$ 的。因此，当雷达平台运动到目标的绝对正侧面（方位坐标 $y$）时，波束中心并没有照射到目标；波束实际照射到目标中心的时间会提前或滞后一段距离 $\Delta y = x \tan(\theta_{r,c})$。
将其除以速度 $V_r$，即得到目标多普勒历程的中心时刻：
$$\eta_c = \frac{y_k - x_k \tan(\theta_{r,c})}{V_r}$$

In [ ]:
# ====================================================================
# 4. 生成点目标原始数据 
# ====================================================================
print(">> 开始生成回波数据 (使用教科书 Sinc^2 天线方向图)...")
s_echo = np.zeros((Naz, Nrg), dtype=np.complex128)
A0 = 1 

for k in range(3):
    # 1. 瞬时斜距
    R_n = np.sqrt(x_range[k]**2 + (Vr * ta_mtx - y_azimuth[k])**2)  
    
    # 2. 距离向包络 (发射脉冲是矩形窗)
    w_range = np.abs(tr_mtx - 2 * R_n / c) <= (Tr / 2)
    
    # 3. 方位向包络 (双程天线方向图，sinc平方型)
    sita = np.arctan(Vr * (ta_mtx - nc_target[k]) / x_range[k])
    w_azimuth_sinc = (np.sinc(0.886 * sita / beta_bw))**2
    w_azimuth_rect = np.abs(ta_mtx - nc_target[k]) <= (La / 2) / Vr
    w_azimuth = w_azimuth_sinc * w_azimuth_rect
    
    # 4. 生成基带回波
    s_k = A0 * w_range * w_azimuth * np.exp(-1j * 4 * np.pi * f0 * R_n / c) * np.exp(1j * np.pi * Kr * (tr_mtx - 2 * R_n / c)**2)
    s_echo += s_k

### 回波生成的四大核心物理模型

1. **瞬时斜距方程**: 描述目标与雷达间距离随方位时间 $\eta$ 的双曲线变化：
$$R(\eta) = \sqrt{x_k^2 + (V_r\eta - y_k)^2}$$

2. **距离向包络**: 控制脉冲宽度，利用矩形窗函数 $\text{rect}$ 限定能量仅在脉冲时宽 $T_r$ 内有效：
$$w_r\left(\tau, \eta\right) = \text{rect}\left(\frac{\tau - 2R(\eta)/c}{T_r}\right)$$

3. **方位向包络**: 模拟真实天线方向图，采用 $\text{sinc}^2$ 函数结合合成孔径长度 $L_a$ 的时间截断：
$$w_a(\eta) = \text{sinc}^2\left(\frac{0.886 \cdot \theta(\eta)}{\beta_{bw}}\right) \cdot \text{rect}\left(\frac{\eta - \eta_c}{L_a/V_r}\right)$$

4. **二维基带回波模型**: 包含传播引起的方位多普勒相位，以及脉冲内部的距离向二次调频相位：
$$s(\tau,\eta) = A_0 \cdot w_r(\tau,\eta) \cdot w_a(\eta) \cdot \exp\left(-j\frac{4\pi f_0 R(\eta)}{c}\right) \cdot \exp\left(j\pi K_r\left(\tau-\frac{2R(\eta)}{c}\right)^2\right)$$

In [ ]:
# ====================================================================
# 5. 距离压缩
# ====================================================================
print(">> 开始距离向脉冲压缩 (512 点补零)...")

# 计算安全的线性卷积长度
N_safe = 512 

# 对回波数据做 512 点 FFT
S_range = np.fft.fft(s_echo, n=N_safe, axis=1)

t_ref = np.arange(-Nr/2, Nr/2) / Fr
t_ref_mtx = np.tile(t_ref, (Naz, 1))

# 引入 Kaiser 窗压制距离向旁瓣
beta = 2.5
win_r = np.kaiser(Nr, beta)
s_ref = np.exp(1j * np.pi * Kr * (t_ref_mtx**2)) * win_r

# 参考脉冲也做 512 点 FFT 变到频域
S_ref = np.fft.fft(s_ref, n=N_safe, axis=1)
H_range = np.conj(S_ref)

# 频域相乘 (线性卷积)
S_range_c = S_range * H_range

# 变回二维时域
s_rc = np.fft.ifft(S_range_c, axis=1)

# 切割有效区域
N_rg = Nrg - Nr + 1
s_rc_c = s_rc[:, :N_rg]

### 距离压缩 (匹配滤波与补零)

* **匹配滤波**: 距离向压缩本质上是接收信号与发射脉冲副本的互相关，在频域表现为相乘：
$$S_{rc}(f_\tau, \eta) = S_{range}(f_\tau, \eta) \cdot H_{ref}^*(f_\tau)$$

* **安全补零 (Zero-padding)**: 离散傅里叶变换计算的是循环卷积。为得到正确的线性卷积并避免混叠，FFT长度 $N_{safe}$ 必须满足：
$$N_{safe} \ge N_{signal} + N_{filter} - 1$$

* **加窗 (Windowing)**: 为了抑制距离压缩后高昂的 $\text{Sinc}$ 函数旁瓣，对频域参考函数施加 Kaiser 窗 $w_{kaiser}$：
$$H_{ref}^*(f_\tau) = \text{FFT}\{ s_{ref}(\tau) \cdot w_{kaiser} \}^*$$

* **卷积完成 (IFFT)**: 
$$s_{rc}(\tau, \eta) = \text{IFFT}\{ S_{rc}(f_\tau, \eta) \}$$

In [ ]:
# ====================================================================
# 6. 变换到距离多普勒域，进行 RCMC
# ====================================================================
print(">> 开始距离多普勒变换与 RCMC (极速 Sinc 插值)...")
# 数据搬移到基带
s_rc_c = s_rc_c * np.exp(-1j * 2 * np.pi * fnc * np.tile(ta[:, np.newaxis], (1, N_rg)))
S_rd = np.fft.fft(s_rc_c, n=NFFT_a, axis=0)

# 方位频率轴
fa = fnc + np.fft.fftshift(np.arange(-NFFT_a/2, NFFT_a/2)) / NFFT_a * Fa

# RCMC 距离时间与随距离门变化的斜距 (空变性修正)
tr_RCMC = 2 * x1 / c + np.arange(-N_rg/2, N_rg/2) / Fr
R0_RCMC = (c / 2) * tr_RCMC * np.cos(sita_r_c)

# 距离徙动量
delta_Rrd_fn = lamda**2 * (fa[:, np.newaxis]**2) * R0_RCMC / (8 * Vr**2)
num_range = c / (2 * Fr)
delta_Rrd_fn_num = delta_Rrd_fn / num_range

# 矢量化 8 点 Sinc 插值 
R = 8
S_rd_rcmc = np.zeros((NFFT_a, N_rg), dtype=np.complex128)

# 计算每个目标像素点的理想浮点坐标
pos = np.arange(N_rg)[np.newaxis, :] + delta_Rrd_fn_num
pos_ceil = np.ceil(pos).astype(int)

weights = np.zeros((R, NFFT_a, N_rg))
pts_wrap = np.zeros((R, NFFT_a, N_rg), dtype=int)

# 生成 8 个加权核
for idx, k in enumerate(range(-R//2 + 1, R//2 + 1)): 
    pts = pos_ceil - k
    weights[idx] = np.sinc(pos - pts) 
    pts_wrap[idx] = pts % N_rg        

# 插值核归一化与并行点乘求和
weights /= np.sum(weights, axis=0)
for idx in range(R):
    S_rd_rcmc += weights[idx] * S_rd[np.arange(NFFT_a)[:, np.newaxis], pts_wrap[idx]]

### 距离徙动校正 (RCMC)

* **基带搬移**: 代码 `s_rc_c * np.exp(-1j * 2 * np.pi * fnc ...)` 用于将数据的多普勒频谱中心强制平移到零频（基带），公式为：
$$s'_{rc}(\tau, \eta) = s_{rc}(\tau, \eta) \cdot \exp(-j 2\pi f_{nc} \eta)$$

* **距离徙动量 ($\Delta R$)**: 在距离多普勒域，点目标的能量轨迹发生弯曲。对于斜距为 $R_0$ 的目标，其徙动量公式为：
$$\Delta R(f_\eta) = \frac{\lambda^2 R_{0}(f_\tau) f_\eta^2}{8 V_r^2}$$
其中，代码利用 `R0_RCMC` 代替固定的中心点斜距，完美考虑了徙动量的空变特性。

* **Sinc 插值核**: 为了以亚像素精度校正偏移，采用长度为 8 的 Sinc 函数进行插值重建：
$$S_{rcmc}(f_\eta, \tau) = \sum_{k=-3}^{4} S_{rd}\left(f_\eta, \tau - \Delta\tau - k\right) \cdot \text{sinc}(k - d)$$
其中，利用模运算 `% N_{rg}` 巧妙处理了 FFT 周期延拓下的边界溢出。

In [ ]:
# ====================================================================
# 7. 方位压缩
# ====================================================================
print(">> 开始方位向压缩...")
fa_azimuth_MF = fa

# 调频率随 R0_RCMC 变化
Ka = 2 * Vr**2 * (np.cos(sita_r_c))**3 / (lamda * R0_RCMC)
Ka_1 = 1 / Ka

# 方位匹配滤波器 
Haz = np.exp(-1j * np.pi * (fa_azimuth_MF[:, np.newaxis]**2) * Ka_1)

S_rd_c = S_rd_rcmc * Haz
s_ac = np.fft.ifft(S_rd_c, axis=0)

### 方位向压缩 (Azimuth Compression)

在消除了距离徙动后，雷达回波在 RD 域的相位仅剩下纯粹的方位向二次调频项。
* **方位调频率 ($K_a$)**: 对于斜视雷达，方位调频率需加入 $\cos^3(\theta_{r,c})$ 修正因子，并且同样随斜距空变：
$$K_a = \frac{2 V_r^2 \cos^3(\theta_{r,c})}{\lambda R_0}$$

* **方位匹配滤波器 ($H_{az}$)**: 在 RD 域与信号相乘即可完成方位聚焦：
$$H_{az}(f_\eta) = \exp\left(-j\pi \frac{f_\eta^2}{K_a}\right)$$

In [ ]:
# ====================================================================
# 8. 统一绘图函数
# ====================================================================
print(">> 渲染图像中...")
def plot_img(data, title, idx):
    plt.figure(idx, figsize=(6, 5))
    plt.imshow(np.abs(data), aspect='auto', cmap='gray')
    plt.title(title)
    plt.xlabel('距离向 (采样点)')
    plt.ylabel('方位向 (采样点)')
    plt.colorbar()
    plt.tight_layout()

plot_img(s_echo, '图1：原始雷达回波数据', 1)
plot_img(s_rc_c, '图2：距离压缩后 (时域)', 2)
plot_img(S_rd,   '图3：距离多普勒域 (未 RCMC)', 3)
plot_img(S_rd_rcmc, '图4：距离多普勒域 (已进行矢量化 Sinc RCMC)', 4)
plot_img(s_ac,   '图5：最终 SAR 聚焦成像', 5)

# ====================================================================
# 9. 小斜视点目标质量分析 (IRW, PSLR, ISLR)
# ====================================================================
print(">> 开始对 3 个点目标进行量化指标评估...")
from scipy.signal import find_peaks

def analyze_1d_target(slice_1d, pixel_spacing, up_factor=16):
    N = len(slice_1d)
    F = np.fft.fftshift(np.fft.fft(np.fft.fftshift(slice_1d)))
    pad_len = N * up_factor
    F_pad = np.pad(F, (pad_len//2 - N//2, pad_len//2 - (N - N//2)), 'constant')
    slice_up = np.abs(np.fft.fftshift(np.fft.ifft(np.fft.fftshift(F_pad)))) * up_factor
    
    res_up = pixel_spacing / up_factor
    
    peak_idx = np.argmax(slice_up)
    peak_val = slice_up[peak_idx]
    slice_norm = slice_up / peak_val
    slice_db = 20 * np.log10(slice_norm + 1e-12)
    
    crossings = np.where(np.diff(np.sign(slice_db + 3)))[0]
    left_crosses = crossings[crossings < peak_idx]
    right_crosses = crossings[crossings >= peak_idx]
    
    if len(left_crosses) == 0 or len(right_crosses) == 0:
        return 0.0, 0.0, 0.0, slice_db, res_up, peak_idx
        
    left_idx, right_idx = left_crosses[-1], right_crosses[0]
    
    x_left = left_idx + (-3 - slice_db[left_idx]) / (slice_db[left_idx+1] - slice_db[left_idx])
    x_right = right_idx + (-3 - slice_db[right_idx]) / (slice_db[right_idx+1] - slice_db[right_idx])
    irw = (x_right - x_left) * res_up
    
    null_width = int((x_right - x_left) * 1.5)
    peaks, _ = find_peaks(slice_norm)
    sidelobes = [p for p in peaks if p < peak_idx - null_width or p > peak_idx + null_width]
    pslr = np.max(slice_db[sidelobes]) if len(sidelobes) > 0 else 0
    
    null_left = max(0, peak_idx - null_width)
    null_right = min(len(slice_norm)-1, peak_idx + null_width)
    E_main = np.sum(slice_norm[null_left:null_right]**2)
    E_total = np.sum(slice_norm**2)
    E_side = E_total - E_main
    islr = 10 * np.log10(E_side / E_main) if E_side > 0 else 0
    
    return irw, pslr, islr, slice_db, res_up, peak_idx

print("\n" + "="*55)
print(" 小斜视点目标成像质量评估报告 ")
print("="*55)

dx = c / (2 * Fr)  
dy = Vr / Fa       

SAR_abs = np.abs(s_ac)
temp_img = SAR_abs.copy()
max_y, max_x = SAR_abs.shape  

WIN = 16 
for target_id in range(3):
    center_y, center_x = np.unravel_index(np.argmax(temp_img), temp_img.shape)
    
    slice_azimuth = SAR_abs[max(0, center_y-WIN):min(max_y, center_y+WIN), center_x]
    slice_range = SAR_abs[center_y, max(0, center_x-WIN):min(max_x, center_x+WIN)]
    
    irw_r, pslr_r, islr_r, db_r, res_r, p_r = analyze_1d_target(slice_range, dx)
    irw_a, pslr_a, islr_a, db_a, res_a, p_a = analyze_1d_target(slice_azimuth, dy)
    
    print(f"▶ 目标 {target_id + 1} (位于矩阵坐标 Y:{center_y}, X:{center_x})")
    print(f"  [距离向 Range]   IRW: {irw_r:.3f} m | PSLR: {pslr_r:.2f} dB | ISLR: {islr_r:.2f} dB")
    print(f"  [方位向 Azimuth] IRW: {irw_a:.3f} m | PSLR: {pslr_a:.2f} dB | ISLR: {islr_a:.2f} dB")
    print("-" * 55)
    
    mask = 8
    temp_img[max(0, center_y-mask):min(max_y, center_y+mask), max(0, center_x-mask):min(max_x, center_x+mask)] = 0

print("✅ 所有目标测试完毕！")
print("="*55 + "\n")
plt.show()

### 图形显示与量化指标评估

为准确评估雷达成像质量，通常需要计算以下三大指标：
1.  **冲激响应宽度 (IRW)**: 主瓣下降到一半能量时的物理宽度（$-3\text{ dB}$）。通过寻峰并在 $-3\text{ dB}$ 处线性插值计算：
$$\text{IRW} = (x_{right} - x_{left}) \cdot \rho$$
2.  **峰值旁瓣比 (PSLR)**: 最高旁瓣能量相对于主瓣最高能量的对数比：
$$\text{PSLR} = 20 \log_{10}\left( \frac{\max(|S_{sidelobe}|)}{\max(|S_{main}|)} \right)$$
3.  **积分旁瓣比 (ISLR)**: 所有旁瓣的总能量与主瓣总能量的比值：
$$\text{ISLR} = 10 \log_{10}\left( \frac{E_{total} - E_{main}}{E_{main}} \right)$$

**注**: 为保证测量精度，提取出的切片会预先在频域补零，执行 **16 倍升采样 (Upsampling)** 还原模拟波形。